# Kaggle: `sft_lora_fp` eval only (adapter already trained)

Training already completed in a prior run (1125/1125 steps, 3h15m, `train_loss: 0.8433`) -- that run then crashed during eval generation on an unrelated dependency bug, not a training problem. This notebook skips training entirely and just evaluates the already-trained adapter, to avoid burning another 3+ hours retraining something that already worked.

**Root cause of the crash**: `peft` 0.20.0 requires `torchao>=0.16.0` for one of its internal LoRA-dispatch checks, but Kaggle's base image ships `torchao==0.10.0`. That check only gets reached when loading a PEFT adapter onto a **full-precision** base model -- `sft_qlora`/`dpo_from_sft` never hit it because loading onto a 4-bit-quantized base goes through a different dispatcher that short-circuits first. Fixed here by uninstalling `torchao` before loading the adapter (this project doesn't use torchao's own quantization scheme at all, only `bitsandbytes` NF4 -- nothing depends on it).

**Before running:**
1. Download the already-trained adapter from the failed run's **Output** tab (`adapters/sft_lora_fp/` -- the crash happened *after* `"Adapter saved to /kaggle/working/adapters/sft_lora_fp"` printed, so it's there) and upload it as a Kaggle Dataset, alongside `data/splits/sft_test.jsonl` and the usual `src.zip` + `config.yaml`.
2. Single T4 accelerator, Internet on.
3. Run all cells.

In [ ]:
!pip install -q -U "trl==1.10.0" "peft==0.20.0" "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" pyyaml
!pip uninstall -y -q torchao
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import os, sys, zipfile

def find_repo(root="/kaggle/input"):
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None or config_path is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir}, config_path={config_path} -- could not find both "
        "src/data/build_irac.py and config.yaml under /kaggle/input (in any "
        "nesting). Check the dataset is attached. If just attached/updated, "
        "try Restart & Run All."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)

import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

MODEL_ID = cfg["model"]["candidates"][cfg["model"]["active"]]["hf_id"]
print("Active model:", MODEL_ID)

In [ ]:
import os

def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

def find_adapter_dir(root="/kaggle/input"):
    """Content-marker search, excluding dry-run artifacts and
    intermediate epoch checkpoints explicitly -- a raw Kaggle output
    upload can contain the dry-run checkpoint, the real adapter's own
    epoch checkpoints (save_strategy="epoch" writes a full adapter to
    output_dir/checkpoint-N/ at every epoch on top of the final state at
    the top level), or both, all matching the same content marker.
    os.walk's traversal order isn't guaranteed alphabetical on Kaggle's
    filesystem, so returning "the first match" could silently grab a
    stale/untrained one with no error. Collects every match and requires
    exactly one non-dryrun, non-checkpoint survivor. See LOG.md
    2026-08-20."""
    candidates = []
    for r, dirs, files in os.walk(root):
        if "adapter_config.json" in files and "adapter_model.safetensors" in files:
            candidates.append(r)
    real = [c for c in candidates if "dryrun" not in c.lower() and "checkpoint-" not in c.lower()]
    if len(real) == 1:
        return real[0]
    raise FileNotFoundError(
        f"Expected exactly one non-dryrun adapter directory under {root}, found {len(real)}: "
        f"{real}. All adapter-like directories found (including dry-run): {candidates}. "
        "Resolve the ambiguity (or absence) before proceeding rather than guessing."
    )

TEST_FILE = find_data_file("sft_test.jsonl")
SFT_LORA_FP_DIR = find_adapter_dir()
print(TEST_FILE, SFT_LORA_FP_DIR, sep="\n")

RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
SUMMARY_CSV = os.path.join(RESULTS_DIR, "summary.csv")

from src.eval.generate import run as generate_run
from src.eval.score import score_file, append_summary_row

In [ ]:
SFT_LORA_FP_GEN = os.path.join(RESULTS_DIR, "sft_lora_fp_gen.jsonl")

generate_run(
    input_path=TEST_FILE,
    output_path=SFT_LORA_FP_GEN,
    model_id=MODEL_ID,
    adapter_path=SFT_LORA_FP_DIR,
    load_in_4bit=False,
    batch_size=16,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
SFT_LORA_FP_SCORED = os.path.join(RESULTS_DIR, "sft_lora_fp_scored.jsonl")
sft_lora_fp_summary = score_file(SFT_LORA_FP_GEN, SFT_LORA_FP_SCORED)
append_summary_row("sft_lora_fp", sft_lora_fp_summary, SUMMARY_CSV)
print("sft_lora_fp:", sft_lora_fp_summary)

In [ ]:
import pandas as pd
df = pd.read_csv(SUMMARY_CSV)
print(df.to_string(index=False))